# FIDE Chess Ratings 2015 to 2026: Starter Notebook

A quick tour of the dataset with efficient loading and a few example analyses.
The file `fide_ratings.csv` holds **130 monthly FIDE standard rating lists** (July 2015 to April 2026),
one row per player per month: about **47.8 million rows** and **575k unique players**.

Use `fide_id` + `snapshot_month` as the unique key to follow a player over time.


## 1. Load the data efficiently

The CSV is ~3.3 GB. Reading it with tight dtypes keeps memory reasonable.
If you only need a few columns, pass `usecols=[...]` to `read_csv` and it loads much faster.
For a single month, filter in chunks instead of loading everything (see the tip at the end).


In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Auto-find the file under /kaggle/input (works whatever the dataset slug is)
matches = glob.glob('/kaggle/input/**/fide_ratings.csv', recursive=True)
CSV = matches[0] if matches else 'fide_ratings.csv'
print('Reading:', CSV)

dtypes = {
    'fide_id': 'int32',
    'name': 'object',
    'sex': 'category',
    'federation': 'category',
    'title': 'category',
    'w_title': 'object',
    'o_title': 'object',
    'rating': 'int16',
    'games': 'int16',
    'k': 'int8',
    'birth_year': 'Int16',
    'flag': 'category',
    'snapshot_month': 'category',
}
df = pd.read_csv(CSV, dtype=dtypes)
print(f'{len(df):,} rows  x  {df.shape[1]} columns')
print(f'memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
df.head()

## 2. What is in here?

Every `(fide_id, snapshot_month)` pair is unique. Ratings are standard (classical) only.


In [ ]:
print('months:', df['snapshot_month'].nunique(),
      '| range:', df['snapshot_month'].min(), 'to', df['snapshot_month'].max())
print('unique players:', df['fide_id'].nunique())
print('federations:', df['federation'].nunique())
df.describe(include='all').T[['count','unique','top','freq']]

## 3. Growth of the rated population

How many rated players appear in each monthly list.


In [ ]:
pop = df.groupby('snapshot_month', observed=True).size()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(pop)), pop.values, color='#3a7d44')
step = 12
ax.set_xticks(range(0, len(pop), step))
ax.set_xticklabels(pop.index[::step], rotation=45, ha='right')
ax.set_title('FIDE rated players per monthly standard list')
ax.set_ylabel('players')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'{pop.iloc[0]:,} players in {pop.index[0]}  ->  {pop.iloc[-1]:,} in {pop.index[-1]}')

## 4. Rating distribution in the latest list

The classic bell-ish shape, floored at 1000.


In [ ]:
latest = sorted(df['snapshot_month'].unique())[-1]
cur = df[df['snapshot_month'] == latest]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cur['rating'], bins=60, color='#3a7d44', alpha=0.85)
ax.set_title(f'Standard rating distribution ({latest}, n={len(cur):,})')
ax.set_xlabel('rating'); ax.set_ylabel('players')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('median rating:', int(cur['rating'].median()))

## 5. Top 10 players in the latest list

Note: the standard list includes inactive players, so retired legends can still appear.
Filter on `flag` (drop `i` and `wi`) if you want active players only.


In [ ]:
cols = ['name', 'federation', 'rating', 'title', 'flag']
cur.nlargest(10, 'rating')[cols].reset_index(drop=True)

## 6. Women and men over time

Share of the rated population by sex, per list. `sex` is the clean field for gender
(the `flag` column also carries a women marker, but `sex` is simpler).


In [ ]:
gt = (df.groupby(['snapshot_month', 'sex'], observed=True).size()
        .unstack(fill_value=0))
share_f = gt.get('F', 0) / gt.sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(share_f)), share_f.values, color='#b5179e')
ax.set_xticks(range(0, len(share_f), 12))
ax.set_xticklabels(share_f.index[::12], rotation=45, ha='right')
ax.set_title('Share of rated players who are women (%)')
ax.set_ylabel('% female'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'women share: {share_f.iloc[0]:.1f}%  ->  {share_f.iloc[-1]:.1f}%')

## 7. Track one player over time

Because the data is a monthly panel, you can plot any player's rating history.
Names are stored as `Surname, Given`.


In [ ]:
player = 'Carlsen, Magnus'
p = df[df['name'] == player].sort_values('snapshot_month')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(p)), p['rating'].values, color='#1d3557')
ax.set_xticks(range(0, len(p), 12))
ax.set_xticklabels(p['snapshot_month'].values[::12], rotation=45, ha='right')
ax.set_title(f'Standard rating over time: {player}')
ax.set_ylabel('rating'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'peak {p.rating.max()}, low {p.rating.min()}, over {len(p)} monthly lists')

## Tips and next ideas

**Memory-light loading.** If the full load is too heavy, read only what you need:

```python
# one column for a time series
pop = pd.read_csv(CSV, usecols=['snapshot_month']).groupby('snapshot_month').size()

# one month, filtered in chunks
parts = []
for chunk in pd.read_csv(CSV, usecols=['name','federation','rating','snapshot_month'],
                         dtype={'rating':'int16'}, chunksize=1_000_000):
    parts.append(chunk[chunk['snapshot_month'] == '2026-04'])
latest = pd.concat(parts, ignore_index=True)
```

**A few caveats to keep in mind**

- Standard ratings only (no rapid or blitz).
- Inactive players are included (`flag` is `i` or `wi`).
- `birth_year` has a small number of placeholders (for example 1900); filter on plausible ages if needed.
- `sex` is blank for a small number of players.

**Ideas to explore:** age vs strength curves, title progression paths, federation growth races,
rating inflation over time, or survival analysis of when players go inactive.

Data: FIDE standard rating lists. Source: https://ratings.fide.com/download_lists.phtml
